# 📊 EDA — Phân tích Dữ liệu Khám phá
**NCKH: AI Recruitment System**

Notebook này phân tích bộ dữ liệu `train.jsonl` và thống kê:
1. Phân phối nhãn NER
2. Độ dài CV (số token)
3. Phân phối kỹ năng phổ biến
4. Thống kê entity theo từng loại


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import Counter, defaultdict
from pathlib import Path

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
plt.style.use('seaborn-v0_8-whitegrid')

TRAIN_FILE = Path('../data/annotated/train.jsonl')
print(f'File exists: {TRAIN_FILE.exists()}')

In [ ]:
# Load dữ liệu
records = []
with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line.strip()))

print(f'Tổng số CV trong dataset: {len(records)}')
print(f'Ví dụ ID: {[r["id"] for r in records[:5]]}')

In [ ]:
# === 1. Thống kê độ dài CV (số token) ===
lengths = [len(r['tokens']) for r in records]
df_len = pd.DataFrame({'cv_id': [r['id'] for r in records], 'n_tokens': lengths})

print('=== Thống kê độ dài CV ===')
print(df_len['n_tokens'].describe().round(1))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lengths, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(pd.Series(lengths).mean(), color='red', linestyle='--', label=f'Mean={pd.Series(lengths).mean():.0f}')
ax.set_xlabel('Số token trong CV')
ax.set_ylabel('Số lượng CV')
ax.set_title('Phân phối Độ dài CV (số token)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# === 2. Phân phối nhãn NER ===
all_tags = []
for r in records:
    all_tags.extend(r['ner_tags'])

tag_counts = Counter(all_tags)

# Chỉ đếm B- tags (entities, không đếm I- và O)
entity_counts = {k.replace('B-', ''): v for k, v in tag_counts.items() if k.startswith('B-')}
entity_df = pd.DataFrame(entity_counts.items(), columns=['entity', 'count']).sort_values('count', ascending=False)

print('=== Phân phối Entity (B-tags) ===')
print(entity_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.tab20.colors
bars = ax.bar(entity_df['entity'], entity_df['count'], color=colors[:len(entity_df)], edgecolor='white')
ax.set_xlabel('Loại Entity')
ax.set_ylabel('Số lần xuất hiện')
ax.set_title('Phân phối Nhãn NER trong Dataset')
plt.xticks(rotation=30, ha='right')
# Thêm số lên đầu cột
for bar, val in zip(bars, entity_df['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# === 3. Tỷ lệ nhãn O vs Entity ===
o_count = tag_counts.get('O', 0)
entity_total = sum(v for k, v in tag_counts.items() if k != 'O')
total = o_count + entity_total

print(f'Tổng tokens: {total:,}')
print(f'  O (non-entity): {o_count:,} ({o_count/total*100:.1f}%)')
print(f'  Entity tokens:  {entity_total:,} ({entity_total/total*100:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie([o_count, entity_total], labels=['Non-Entity (O)', 'Entity tokens'],
       autopct='%1.1f%%', colors=['#95a5a6', '#3498db'], startangle=90)
ax.set_title('Tỷ lệ Token Entity vs Non-Entity')
plt.tight_layout()
plt.show()

In [ ]:
# === 4. Trích xuất và đếm các SKILL phổ biến ===
skill_counter = Counter()
for r in records:
    tokens = r['tokens']
    tags = r['ner_tags']
    current_skill = []
    for tok, tag in zip(tokens, tags):
        if tag == 'B-SKILL':
            if current_skill:
                skill_counter[' '.join(current_skill)] += 1
            current_skill = [tok]
        elif tag == 'I-SKILL' and current_skill:
            current_skill.append(tok)
        else:
            if current_skill:
                skill_counter[' '.join(current_skill)] += 1
            current_skill = []
    if current_skill:
        skill_counter[' '.join(current_skill)] += 1

top_skills = skill_counter.most_common(25)
print(f'=== Top 25 Kỹ năng Phổ biến Nhất ===')
for skill, cnt in top_skills:
    print(f'  {skill}: {cnt}')

skill_df = pd.DataFrame(top_skills, columns=['skill', 'count'])
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(skill_df['skill'][::-1], skill_df['count'][::-1], color='#2ecc71', edgecolor='white')
ax.set_xlabel('Số CV có kỹ năng này')
ax.set_title('Top 25 Kỹ năng Xuất hiện Nhiều Nhất')
for bar, val in zip(bars, skill_df['count'][::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# === 5. Số entity trung bình per CV ===
entity_stats = defaultdict(list)
for r in records:
    b_counts = Counter(t for t in r['ner_tags'] if t.startswith('B-'))
    for entity_type in ['SKILL', 'JOB_TITLE', 'UNIVERSITY', 'DEGREE', 'COMPANY', 'DURATION', 'GPA', 'SOFT_SKILL']:
        entity_stats[entity_type].append(b_counts.get(f'B-{entity_type}', 0))

stats_df = pd.DataFrame({
    ent: {'mean': pd.Series(vals).mean(), 'max': max(vals), 'min': min(vals)}
    for ent, vals in entity_stats.items()
}).T.round(2)

print('=== Số entity trung bình mỗi CV ===')
print(stats_df.to_string())

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(stats_df))
ax.bar(x, stats_df['mean'], color='#9b59b6', edgecolor='white', alpha=0.8, label='Mean')
ax.set_xticks(list(x))
ax.set_xticklabels(stats_df.index, rotation=30, ha='right')
ax.set_ylabel('Số entity trung bình')
ax.set_title('Số Entity Trung bình Mỗi CV theo Loại')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# === 6. Heatmap: CV x Entity Type ===
cv_entity_matrix = []
entity_types = ['SKILL', 'JOB_TITLE', 'UNIVERSITY', 'DEGREE', 'COMPANY', 'DURATION', 'GPA', 'SOFT_SKILL', 'PROJECT_NAME']

for r in records:
    b_counts = Counter(t for t in r['ner_tags'] if t.startswith('B-'))
    row = {ent: b_counts.get(f'B-{ent}', 0) for ent in entity_types}
    row['cv_id'] = r['id']
    cv_entity_matrix.append(row)

mat_df = pd.DataFrame(cv_entity_matrix).set_index('cv_id')

if len(mat_df) <= 50:  # Chỉ hiển thị heatmap nếu dataset không quá lớn
    fig, ax = plt.subplots(figsize=(14, max(6, len(mat_df) * 0.3)))
    sns.heatmap(mat_df, annot=True, fmt='d', cmap='Blues', linewidths=0.5, ax=ax)
    ax.set_title('Heatmap: Số Entity theo CV và Loại')
    ax.set_xlabel('Loại Entity')
    ax.set_ylabel('CV ID')
    plt.tight_layout()
    plt.show()
else:
    print(f'Dataset có {len(mat_df)} CV — bỏ qua heatmap để tránh quá tải.')
    print(mat_df.describe().round(1))

In [ ]:
# === 7. Tổng kết ===
print('='*50)
print('📊 TỔNG KẾT DATASET')
print('='*50)
print(f'  Tổng số CV: {len(records)}')
print(f'  Tổng tokens: {sum(lengths):,}')
print(f'  Độ dài TB: {sum(lengths)/len(lengths):.0f} tokens/CV')
print(f'  Tổng entity instances: {entity_total:,}')
print(f'  Số loại entity: {len(entity_counts)}')
print(f'  Tổng skill unique: {len(skill_counter)}')
print('='*50)